In [45]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [47]:

print("  SUBSCRIPTION DATA — PREPARATION PIPELINE")


  SUBSCRIPTION DATA — PREPARATION PIPELINE


In [49]:
df = pd.read_csv("D:\Work\Exccel\Streaming Video Subscription\Raw Dataset\subscriptions_raw.csv")

In [50]:
print(f"\n✓ Raw data loaded")
print(f"  Rows    : {df.shape[0]}")
print(f"  Columns : {df.shape[1]}")
print(f"\n  Columns found : {list(df.columns)}")


✓ Raw data loaded
  Rows    : 3069
  Columns : 6

  Columns found : ['customer_id', 'created_date', 'canceled_date', 'subscription_cost', 'subscription_interval', 'was_subscription_paid']


In [51]:
print(df.head())

   customer_id created_date canceled_date  subscription_cost  \
0    154536156   2022-09-01           NaN                 39   
1    149713408   2022-09-01    2022-09-02                 39   
2    153756284   2022-09-01    2022-09-02                 39   
3    121253113   2022-09-01    2022-09-23                 39   
4    154467210   2022-09-01    2023-06-29                 39   

  subscription_interval was_subscription_paid  
0                 month                   Yes  
1                 month                    No  
2                 month                    No  
3                 month                   Yes  
4                 month                   Yes  


In [52]:
print(df.dtypes)

customer_id               int64
created_date             object
canceled_date            object
subscription_cost         int64
subscription_interval    object
was_subscription_paid    object
dtype: object


In [53]:
print(df.isnull().sum())

customer_id                 0
created_date                0
canceled_date            1065
subscription_cost           0
subscription_interval       0
was_subscription_paid       0
dtype: int64


In [54]:
df['created_date']   = pd.to_datetime(df['created_date'],   errors='coerce')
df['canceled_date']  = pd.to_datetime(df['canceled_date'],  errors='coerce')
df['subscription_cost'] = pd.to_numeric(df['subscription_cost'], errors='coerce')
df['customer_id']    = df['customer_id'].astype(str).str.strip()
df['was_subscription_paid'] = df['was_subscription_paid'].astype(str).str.strip().str.lower()
df['subscription_interval'] = df['subscription_interval'].astype(str).str.strip().str.lower()

In [58]:
df['subscription_status'] = df['canceled_date'].apply(
    lambda x: 'Cancelled' if pd.notna(x) else 'Active'
)
print("✓ subscription_status column added")

✓ subscription_status column added


In [57]:
df['churn_flag'] = df['canceled_date'].apply(
    lambda x: 1 if pd.notna(x) else 0
)
print("✓ churn_flag columnn added")

✓ churn_flag columnn added


In [59]:
df['active_flag'] = df['canceled_date'].apply(
    lambda x: 0 if pd.notna(x) else 1
)
print("✓ active_flag column added")

✓ active_flag column added


In [63]:
def get_monthly_revenue(row):
    cost     = row['subscription_cost']
    interval = str(row['subscription_interval']).strip().lower()
    if   interval in ['month', 'monthly']:     return round(cost, 2)
    elif interval in ['year',  'yearly', 'annual']: return round(cost / 12, 2)
    elif interval in ['week',  'weekly']:       return round(cost * 4.33, 2)
    elif interval in ['quarter','quarterly']:   return round(cost / 3, 2)
    else:                                        return round(cost, 2)
 
df['monthly_revenue'] = df.apply(get_monthly_revenue, axis=1)
print("✓ monthly_revenue column added")

✓ monthly_revenue column added


In [64]:
df['Revenue'] = df.apply(
    lambda x: round(x['monthly_revenue'], 2)
    if str(x['was_subscription_paid']).strip().lower() == 'true'
    else 0.0,
    axis=1
)
print("✓ Revenue added (paid only)")

✓ Revenue added (paid only)


In [65]:
def get_duration(row):
    try:
        end   = row['canceled_date'] if pd.notna(row['canceled_date']) else datetime.now()
        start = row['created_date']
        months = (end.year - start.year) * 12 + (end.month - start.month)
        return max(months, 0)
    except:
        return 0
 
df['subscription_duration'] = df.apply(get_duration, axis=1)
print("✓ subscription_duration column added")

✓ subscription_duration column added


In [66]:
df['clv_per_customer'] = round(df['monthly_revenue'] * df['subscription_duration'], 2)
print("✓ clv_per_customer column added")

✓ clv_per_customer column added


In [67]:
df['cohort_month'] = df['created_date'].dt.strftime('%Y-%m')
print("✓ cohort_month column added")

✓ cohort_month column added


In [70]:
df['cancel_month'] = df['canceled_date'].apply(
    lambda x: x.strftime('%Y-%m') if pd.notna(x) else None
)
print("✓ cancel_month column added")

✓ cancel_month column added


In [71]:
df['months_since_join'] = df['subscription_duration'].copy()
print("✓ months_since_join column added")

✓ months_since_join column added


In [72]:
print("\n── Final Column List ────────────────────────────────")
for col in df.columns:
    print(f"  {col:30s}  {str(df[col].dtype):15s}  nulls: {df[col].isnull().sum()}")


── Final Column List ────────────────────────────────
  customer_id                     object           nulls: 0
  created_date                    datetime64[ns]   nulls: 0
  canceled_date                   datetime64[ns]   nulls: 1065
  subscription_cost               int64            nulls: 0
  subscription_interval           object           nulls: 0
  was_subscription_paid           object           nulls: 0
  subscription_status             object           nulls: 0
  churn_flag                      int64            nulls: 0
  active_flag                     int64            nulls: 0
  monthly_revenue                 int64            nulls: 0
  Revenue                         float64          nulls: 0
  subscription_duration           int64            nulls: 0
  clv_per_customer                int64            nulls: 0
  cohort_month                    object           nulls: 0
  cancel_month                    object           nulls: 1065
  months_since_join               int64

In [73]:
print("\n── Quick Summary ────────────────────────────────────")
print(f"  Total customers      : {len(df):,}")
print(f"  Active customers     : {df['active_flag'].sum():,}")
print(f"  Churned customers    : {df['churn_flag'].sum():,}")
print(f"  Churn rate           : {df['churn_flag'].mean()*100:.1f}%")
print(f"  Total MRR            : ${df['Revenue'].sum():,.2f}")
print(f"  Avg CLV              : ${df['clv_per_customer'].mean():,.2f}")
print(f"  Avg duration (months): {df['subscription_duration'].mean():.1f}")
 


── Quick Summary ────────────────────────────────────
  Total customers      : 3,069
  Active customers     : 1,065
  Churned customers    : 2,004
  Churn rate           : 65.3%
  Total MRR            : $0.00
  Avg CLV              : $521.59
  Avg duration (months): 13.4


In [74]:
df.to_csv('customer_clean.csv', index=False)
print("\n✓ Clean CSV exported → customer_clean.csv")
print("  Load this file in Excel Power Query for dashboard")


✓ Clean CSV exported → customer_clean.csv
  Load this file in Excel Power Query for dashboard
